# Cleaned Difference Heatmaps With Confidence Intervals

This notebook recreates the cleaned-analysis weather heatmaps for each team, but only for the `difference from team mean` version.

Each heatmap cell shows:

- the bucket mean minus the team-wide mean for that baseball stat
- a bootstrap 68% confidence interval for that difference

Outputs are saved to `analysis/cleaned_analysis_diff_ci/<TEAM>/`.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import TwoSlopeNorm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

# ============================================================
# PARAMETERS
# ============================================================
TEAM = 'SF'
RUN_ALL_TEAMS = True
SEASON_START = None
SEASON_END = None

DATASETS_DIR = os.path.join('..', 'data')
OUTPUT_ROOT = 'cleaned_analysis_diff_ci'
SUMMARY_FILE = os.path.join(OUTPUT_ROOT, 'heatmap_ci_summary.csv')

N_BOOTSTRAP = 1000
CI_LEVEL = 0.68
RANDOM_SEED = 192

params_df = pd.read_csv('team_parameters.csv')
os.makedirs(OUTPUT_ROOT, exist_ok=True)

teams_to_process = params_df['team_code'].tolist() if RUN_ALL_TEAMS else [TEAM]
print(f'Teams to process: {teams_to_process}')
print(f'Output root: {os.path.abspath(OUTPUT_ROOT)}')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
WEATHER_VARS = {
    'temp_f': {'label': 'Temperature', 'unit': ' F', 'n_bins': 5, 'fmt': '.0f'},
    'wspd_mph': {'label': 'Wind Speed', 'unit': ' mph', 'n_bins': 5, 'fmt': '.0f'},
    'rhum': {'label': 'Humidity', 'unit': '%', 'n_bins': 5, 'fmt': '.0f'},
    'pres': {'label': 'Pressure', 'unit': ' hPa', 'n_bins': 5, 'fmt': '.0f'},
}

BASEBALL_STATS = {
    'away_runs_scored': {'label': 'Away Runs', 'fmt': '.2f'},
    'away_bat_hr': {'label': 'Away Home Runs', 'fmt': '.2f'},
    'away_bat_k': {'label': 'Away Strikeouts', 'fmt': '.2f'},
    'away_bat_bb': {'label': 'Away Walks', 'fmt': '.2f'},
    'away_bat_hr_h_ratio': {'label': 'Away HR:H Ratio', 'fmt': '.3f'},
}

print('Configuration loaded.')


def day_night_dataset_file(dataset_file):
    base, ext = os.path.splitext(dataset_file)
    return f'{base}_day_night{ext}'


def prepare_team_data(team_row):
    dataset_file = day_night_dataset_file(team_row['dataset_file'])
    dataset_path = os.path.join(DATASETS_DIR, dataset_file)
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f'Dataset not found: {dataset_path}')

    df = pd.read_csv(dataset_path)
    df['game_date'] = pd.to_datetime(df['game_date'])

    if 'day_night' not in df.columns:
        if 'start_hour' not in df.columns:
            raise ValueError(f'{dataset_file} is missing both day_night and start_hour')
        df['day_night'] = np.where(pd.to_numeric(df['start_hour'], errors='coerce') < 17, 'day', 'night')

    season_start = int(team_row['data_start_year']) if SEASON_START is None else int(SEASON_START)
    season_end = int(team_row['data_end_year']) if SEASON_END is None else int(SEASON_END)
    df = df[(df['season'] >= season_start) & (df['season'] <= season_end)].copy()
    return df, dataset_file, season_start, season_end


def bucket_weather_var(df, col, config):
    series = df[col].dropna()
    q = min(config['n_bins'], max(1, series.nunique()))
    if len(series) == 0 or q < 2:
        return None, [], {}

    buckets = pd.qcut(series, q=q, duplicates='drop')
    label_map = {}
    for interval in buckets.cat.categories:
        left = format(interval.left, config['fmt'])
        right = format(interval.right, config['fmt'])
        label_map[interval] = f'{left}-{right}{config["unit"]}'

    labeled = buckets.map(label_map)
    bucket_order = [label_map[iv] for iv in buckets.cat.categories]
    counts = labeled.value_counts().reindex(bucket_order).fillna(0).astype(int).to_dict()
    return labeled, bucket_order, counts


def bootstrap_diff_ci(bucket_values, overall_mean, rng, n_bootstrap=N_BOOTSTRAP, ci_level=CI_LEVEL):
    bucket_values = np.asarray(bucket_values, dtype=float)
    bucket_values = bucket_values[np.isfinite(bucket_values)]
    if len(bucket_values) == 0:
        return np.nan, np.nan

    if len(bucket_values) == 1:
        diff = bucket_values[0] - overall_mean
        return diff, diff

    samples = rng.choice(bucket_values, size=(n_bootstrap, len(bucket_values)), replace=True)
    boot_diffs = samples.mean(axis=1) - overall_mean
    alpha = 1.0 - ci_level
    lower = np.quantile(boot_diffs, alpha / 2)
    upper = np.quantile(boot_diffs, 1 - alpha / 2)
    return lower, upper


In [ ]:
def build_heatmap_payload(df, weather_col, weather_config, rng):
    labeled, bucket_order, counts = bucket_weather_var(df, weather_col, weather_config)
    if labeled is None or len(bucket_order) == 0:
        return None

    stat_keys = list(BASEBALL_STATS.keys())
    stat_labels = [BASEBALL_STATS[k]['label'] for k in stat_keys]

    df_valid = df.loc[labeled.index].copy()
    df_valid['_bucket'] = labeled.values

    value_matrix = np.full((len(stat_keys), len(bucket_order)), np.nan)
    lower_matrix = np.full((len(stat_keys), len(bucket_order)), np.nan)
    upper_matrix = np.full((len(stat_keys), len(bucket_order)), np.nan)
    records = []

    for i, stat in enumerate(stat_keys):
        overall_mean = df[stat].mean()
        if pd.isna(overall_mean):
            continue

        for j, bucket_label in enumerate(bucket_order):
            bucket_values = df_valid.loc[df_valid['_bucket'] == bucket_label, stat].dropna().to_numpy()
            if len(bucket_values) == 0:
                continue

            diff_value = bucket_values.mean() - overall_mean
            ci_low, ci_high = bootstrap_diff_ci(bucket_values, overall_mean, rng)

            value_matrix[i, j] = diff_value
            lower_matrix[i, j] = ci_low
            upper_matrix[i, j] = ci_high

            records.append({
                'weather_var': weather_col,
                'weather_bucket': bucket_label,
                'weather_bucket_games': counts.get(bucket_label, 0),
                'stat': stat,
                'stat_label': BASEBALL_STATS[stat]['label'],
                'team_mean': overall_mean,
                'bucket_mean': bucket_values.mean(),
                'difference_from_team_mean': diff_value,
                'ci_lower': ci_low,
                'ci_upper': ci_high,
                'n_stat_games': len(bucket_values),
            })

    if np.isnan(value_matrix).all():
        return None

    return {
        'bucket_order': bucket_order,
        'counts': counts,
        'stat_keys': stat_keys,
        'stat_labels': stat_labels,
        'value_matrix': value_matrix,
        'lower_matrix': lower_matrix,
        'upper_matrix': upper_matrix,
        'records': records,
    }


def render_diff_heatmap(df, weather_col, weather_config, stadium_name, season_start, season_end, filepath, rng, show=True):
    payload = build_heatmap_payload(df, weather_col, weather_config, rng)
    if payload is None:
        return None, []

    matrix = payload['value_matrix']
    finite_vals = matrix[np.isfinite(matrix)]
    bound = max(abs(np.nanmin(finite_vals)), abs(np.nanmax(finite_vals)))
    if bound == 0:
        bound = 1e-9

    norm = TwoSlopeNorm(vmin=-bound, vcenter=0, vmax=bound)
    fig_width = max(8, len(payload['bucket_order']) * 1.85)
    fig_height = max(5.2, len(payload['stat_labels']) * 0.95)
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    im = ax.imshow(matrix, cmap=plt.cm.RdBu_r, norm=norm, aspect='auto')

    ax.set_xticks(np.arange(len(payload['bucket_order'])))
    ax.set_yticks(np.arange(len(payload['stat_labels'])))
    ax.set_xticklabels([f'{label}\n(n={payload["counts"].get(label, 0)})' for label in payload['bucket_order']], fontsize=9)
    ax.set_yticklabels(payload['stat_labels'], fontsize=10)

    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if not np.isfinite(matrix[i, j]):
                continue
            ci_low = payload['lower_matrix'][i, j]
            ci_high = payload['upper_matrix'][i, j]
            mean_text = f'{matrix[i, j]:.2f}'
            ci_text = f'[{ci_low:.2f}, {ci_high:.2f}]'
            ax.text(j, i - 0.12, mean_text, ha='center', va='center', fontsize=9.5, fontweight='semibold', color='black')
            ax.text(j, i + 0.18, ci_text, ha='center', va='center', fontsize=6.5, color='black')

    ax.set_title(
        f'{weather_config["label"]} Heatmap (Difference from Team Mean)\n{stadium_name} ({season_start}-{season_end})',
        fontsize=13,
    )
    cbar = fig.colorbar(im, ax=ax, shrink=0.9)
    cbar.set_label('Difference from Team Mean')
    plt.tight_layout()
    fig.savefig(filepath, dpi=150, bbox_inches='tight', facecolor='white')
    if show:
        plt.show()
    else:
        plt.close(fig)
    return filepath, payload['records']


def run_team_heatmap_analysis(team_code, params_df, show=True):
    team_row = params_df[params_df['team_code'] == team_code]
    if len(team_row) == 0:
        return {'team': team_code, 'status': 'NOT FOUND', 'n_games': 0, 'n_graphics': 0}, []

    team_row = team_row.iloc[0]
    team_name = team_row['team_name']
    stadium_name = team_row['stadium_name']
    team_dir = os.path.join(OUTPUT_ROOT, team_code)
    os.makedirs(team_dir, exist_ok=True)

    team_data, dataset_file, season_start, season_end = prepare_team_data(team_row)
    print('\n' + '=' * 60)
    print(f'  {stadium_name} ({team_name}) - {team_code}')
    print(f'  Seasons: {season_start}-{season_end}')
    print(f'  Dataset: {dataset_file}')
    print(f'  Output: {os.path.abspath(team_dir)}')
    print(f'  Games: {len(team_data)}')
    print('=' * 60)

    rng = np.random.default_rng(RANDOM_SEED)
    n_graphics = 0
    team_records = []

    for weather_col, weather_config in WEATHER_VARS.items():
        filepath = os.path.join(team_dir, f'heatmap_{weather_col}_diff_ci.png')
        output, records = render_diff_heatmap(
            team_data,
            weather_col,
            weather_config,
            stadium_name,
            season_start,
            season_end,
            filepath,
            rng,
            show=show,
        )
        for record in records:
            record.update({
                'team': team_code,
                'team_name': team_name,
                'stadium_name': stadium_name,
                'dataset_file': dataset_file,
                'season_start': season_start,
                'season_end': season_end,
            })
        team_records.extend(records)
        if output is not None:
            n_graphics += 1

    return {
        'team': team_code,
        'stadium': stadium_name,
        'status': 'OK',
        'dataset': dataset_file,
        'n_games': len(team_data),
        'n_graphics': n_graphics,
    }, team_records


print('Helper functions defined.')

In [ ]:
show_plots = not RUN_ALL_TEAMS
if RUN_ALL_TEAMS:
    plt.ioff()
    print(f'Batch mode: generating heatmaps for {len(teams_to_process)} teams')

results_summary = []
all_records = []

for i, team_code in enumerate(teams_to_process, start=1):
    if RUN_ALL_TEAMS:
        print(f'\n[{i}/{len(teams_to_process)}] Processing {team_code}...')
    try:
        summary_row, team_records = run_team_heatmap_analysis(team_code, params_df, show=show_plots)
        results_summary.append(summary_row)
        all_records.extend(team_records)
    except Exception as exc:
        print(f'  ERROR processing {team_code}: {exc}')
        results_summary.append({
            'team': team_code,
            'stadium': None,
            'status': f'ERROR: {exc}',
            'dataset': None,
            'n_games': 0,
            'n_graphics': 0,
        })

summary_df = pd.DataFrame(results_summary)
summary_df

if all_records:
    detail_df = pd.DataFrame(all_records)
    detail_df.to_csv(SUMMARY_FILE, index=False)
    print(f'Detailed heatmap summary saved to: {os.path.abspath(SUMMARY_FILE)}')
else:
    print('No detailed heatmap records were generated.')